In [ ]:
!pip freeze

absl-py==1.4.0
absolufy-imports==0.3.1
accelerate==1.11.0
aiofiles==24.1.0
aiohappyeyeballs==2.6.1
aiohttp==3.13.1
aiosignal==1.4.0
alabaster==1.0.0
albucore==0.0.24
albumentations==2.0.8
ale-py==0.11.2
alembic==1.17.0
altair==5.5.0
annotated-doc==0.0.3
annotated-types==0.7.0
antlr4-python3-runtime==4.9.3
anyio==4.11.0
anywidget==0.9.18
argon2-cffi==25.1.0
argon2-cffi-bindings==25.1.0
array_record==0.8.2
arrow==1.4.0
arviz==0.22.0
astropy==7.1.1
astropy-iers-data==0.2025.10.27.0.39.10
astunparse==1.6.3
atpublic==5.1
attrs==25.4.0
audioread==3.1.0
Authlib==1.6.5
autograd==1.8.0
babel==2.17.0
backcall==0.2.0
beartype==0.22.4
beautifulsoup4==4.13.5
betterproto==2.0.0b6
bigframes==2.27.0
bigquery-magics==0.10.3
bleach==6.3.0
blinker==1.9.0
blis==1.3.0
blobfile==3.1.0
blosc2==3.11.0
bokeh==3.7.3
Bottleneck==1.4.2
bqplot==0.12.45
branca==0.8.2
Brotli==1.1.0
build==1.3.0
CacheControl==0.14.3
cachetools==5.5.2
catalogue==2.0.10
certifi==2025.10.5
cffi==2.0.0
chardet==5.2.0
charset-normalizer==

# Setup

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
# import libraries
import os, base64
import pandas as pd
from openai import OpenAI
from pydantic import BaseModel

In [ ]:
# Working directory
directory = "/content/drive/MyDrive/GenAI/RAG with OpenAI API/OpenAI Project"
os.chdir(directory)

In [ ]:
# Retrieve the OpenAI API key and set in the os
from google.colab import userdata
openai_api_key = userdata.get('OPENAI_API_KEY')
os.environ['OPENAI_API_KEY'] = openai_api_key

In [ ]:
# Connect to the api and set the gpt model
client = OpenAI()
MODEL = "gpt-5.6-luna"

In [ ]:
# select the folder where the images are
images_folder = input("Enter the name of the folder: ")
IMAGE_DIR = os.path.join(directory, images_folder)
print(f"The images are in {IMAGE_DIR}")

Enter the name of the folder: Regatta Folder
The images are in /content/drive/MyDrive/GenAI/RAG with OpenAI API/OpenAI Project/Regatta Folder


# Images

In [ ]:
# Build a function that converts the images into base 64
def encode_image(image_path):
  with open(image_path, "rb") as file:
    b64 = base64.b64encode(file.read()).decode("utf-8")
  return f"data:image/jpeg;base64,{b64}"

In [ ]:
# Get the images
image_files = sorted(
    [file for file in os.listdir(IMAGE_DIR) if file.endswith((".jpg",".png", "jpeg"))]
)
image_files

['Regatta_page_1.jpg',
 'Regatta_page_2.jpg',
 'Regatta_page_3.jpg',
 'Regatta_page_4.jpg',
 'Regatta_page_5.jpg',
 'Regatta_page_6.jpg',
 'Regatta_page_7.jpg']

# Structured Format

In [ ]:
# Build a Class for the MenuItem
class MenuItem(BaseModel):
  CategoryTitleDefault: str = ""
  SubcategoryTitleDefault: str = ""
  ItemNameDefault: str = ""
  ItemDescriptionDefault: str = ""
  ItemPrice: str = ""

In [ ]:
# Extract the columns
COLUMNS = list(MenuItem.model_fields.keys())
COLUMNS

['CategoryTitleDefault',
 'SubcategoryTitleDefault',
 'ItemNameDefault',
 'ItemDescriptionDefault',
 'ItemPrice']

In [ ]:
# Build a PayLoad class
class MenuPayLoad(BaseModel):
  items: list[MenuItem]
MenuPayLoad

__main__.MenuPayLoad

# System Prompt

In [ ]:
# Define the System Prompt
system_prompt = """
You convert menu images into a structured Excel-ready dataset with EXACTLY these 5 columns.

Rules:
- Each row is one item.
- Leave fields as "" if unknown or not shown.
- Append only; do not remove anything.
- Output ONLY JSON that matches the schema exactly.

Columns and limits:
- CategoryTitleDefault: text, max 30 chars
- SubcategoryTitleDefault: text, max 30 chars ("" if none)
- ItemNameDefault: text, max 40 chars
- ItemDescriptionDefault: text, max 300 chars ("" if none)
- ItemPrice: decimal number as text, no currency symbol (e.g., "2.50" or "2,50")
"""

In [ ]:
# Test the API call with 1 image
rows = []
for image in image_files:
  try:
    image_url = encode_image(os.path.join(IMAGE_DIR, image))
    response = client.responses.parse(
        model = MODEL,
        instructions = system_prompt,
        input = [
            {
            "role": "user", "content": [
                {"type": "input_text", "text": "extract menu images from this image. Output must match the schema"},
                {"type": "input_image", "image_url": image_url}
            ]
        }],
        text_format = MenuPayLoad
    )
    payload = response.output_parsed
    rows.extend([item.model_dump() for item in payload.items])
    print(f"{image}: {len(payload.items)} item(s)")

  except Exception as e:
    print(f"{image}: {e}")

# display total amount of items transcribed
print(f"Total items: {len(rows)}")

Regatta_page_1.jpg: 13 item(s)
Regatta_page_2.jpg: 5 item(s)
Regatta_page_3.jpg: 15 item(s)
Regatta_page_4.jpg: 2 item(s)
Regatta_page_5.jpg: 8 item(s)
Regatta_page_6.jpg: 22 item(s)


In [ ]:
# Transform the output into a df
df = pd.DataFrame(rows, columns = COLUMNS)
df

,CategoryTitleDefault,SubcategoryTitleDefault,ItemNameDefault,ItemDescriptionDefault,ItemPrice
0,Entradas / Starters,,Azeitonas,Olives,1.60
1,Entradas / Starters,,Cesto de Pão,Bread Basket,1.80
2,Entradas / Starters,,Creme de Legumes,Cream of Vegetables,3.00
3,Para Partilhar / To Share,,Nachos & Guacamole,,7.50
4,Para Partilhar / To Share,,REGATTA Tuna,Atum Fresco Marinado / Marinated Tuna,12.90
...,...,...,...,...,...
76,DIGESTIVOS & LICORES,,Moscatel Favaios,,2.00
77,DIGESTIVOS & LICORES,,Licor Beirão,,3.00
78,DIGESTIVOS & LICORES,,Vinho do Porto,,3.00
79,DIGESTIVOS & LICORES,,Whisky,,5.00


In [ ]:
# Ask for a name for the excel file
excel_name = input("Enter the name of the excel file: ")
# Add the .xlsx extension if it's not already present
if not excel_name.endswith('.xlsx'):
    excel_name += '.xlsx'
df.to_excel(excel_name, index = False)

Enter the name of the excel file: regatta
